# Ayas: compile the defect YOLOv8n to a Hailo-10H HEF (Colab)

**The Pi fixes the versions, not what is newest.** The Pi runs HailoRT **5.1.1** (apt package `hailo-h10-all`). Compile only with the **Dataflow Compiler (DFC) 5.1.0** and the **Hailo Model Zoo v5.1.0**: a HEF from a newer DFC (5.5) may not load. The model is **YOLOv8n** (YOLO26 has no v5.1.0 support).

What this notebook does:
1. Checks the runtime (T4 GPU, Python 3.12) and mounts Drive.
2. Installs the DFC 5.1.0 wheel you downloaded, and the Model Zoo v5.1.0.
3. **Step 0, go/no-go 1:** compiles the stock COCO yolov8n, and saves Hailo's precompiled v5.1.0 yolov8n HEF as a known-good reference. Test both on the Pi before going on.
4. Compiles the Ayas steel-surface model (`best.onnx`, 6 classes: scratch, dent, pit, corrosion, crack, stain = `ayas.tiles.CLASSES`) with 1,024 calibration tiles, checks that every output's SNR is at least 10 dB, and saves the HEF and HAR to Drive.

**Put these in `MyDrive/ayas_hailo/` first:**
- `hailo_dataflow_compiler-5.1.0-*.whl`: download it yourself from the Hailo Developer Zone (log in in your own browser; never type credentials into this notebook).
- `best.onnx`: from `training/export_onnx.py` (640 px, opset 11, no NMS).
- `calib_640.zip`: the `calib_640/` folder from `training/calib_set.py`: TRAIN tiles of the burst pipeline (real burst captures; in Phase 1 the twin burst-arm tiles with `--allow-twin`), never a test plate (P5, lighting-matrix seeds >= 900000). On the Mac: `cd runs/calib && zip -qr calib_640.zip calib_640`.

## 1. Runtime and Drive

**Runtime > Change runtime type:** Hardware accelerator **T4 GPU**, Runtime version **2026.07** (Python 3.12). The default runtime has Python 3.13, which DFC 5.1.0 rejects. Without a GPU the optimisation level drops to 0.

In [ ]:
import subprocess
import sys

print(sys.version)
assert sys.version_info[:2] == (3, 12), "DFC 5.1.0 needs Python 3.12: Runtime > Change runtime type > Runtime version 2026.07"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout or "NO GPU: switch to a T4 runtime (optimisation would drop to level 0)")

In [ ]:
import glob
import os

from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/ayas_hailo"      # inputs and outputs live here
OUT = f"{WORK}/out"
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir(WORK)))

## 2. Install the DFC 5.1.0 and the Model Zoo v5.1.0

In [ ]:
!apt-get -qq update && apt-get -qq install -y graphviz graphviz-dev > /dev/null
wheels = sorted(glob.glob(f"{WORK}/hailo_dataflow_compiler-5.1.0*.whl"))
assert wheels, f"put the DFC 5.1.0 wheel from the Hailo Developer Zone in {WORK}"
WHL = wheels[0]
!pip install -q "{WHL}"
from importlib.metadata import version

DFC = version("hailo-dataflow-compiler")
print("DFC", DFC)
assert DFC.startswith("5.1.0"), DFC

In [ ]:
%cd /content
!rm -rf hailo_model_zoo && git clone -q -b v5.1.0 --depth 1 https://github.com/hailo-ai/hailo_model_zoo.git
!pip install -q -e /content/hailo_model_zoo
!git -C /content/hailo_model_zoo describe --tags
!hailomz --help | head -n 5
assert version("hailo-dataflow-compiler") == DFC, "the Model Zoo install changed the DFC version"

## 3. Step 0 (go/no-go 1): the stock COCO yolov8n

Prove the chain before touching the custom model: a HEF built by this DFC must load and run on the Pi's HailoRT 5.1.1. Calibration quality does not matter for this check, so 1,024 COCO val2017 images are used. Hailo's precompiled v5.1.0 HEF is saved next to it as the known-good reference.

In [ ]:
import shutil
import urllib.request

REF = "https://hailo-model-zoo.s3.eu-west-2.amazonaws.com/ModelZoo/Compiled/v5.1.0/hailo10h/yolov8n.hef"
urllib.request.urlretrieve(REF, f"{OUT}/yolov8n_hailo_v5.1.0_reference.hef")

!wget -q -nc http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!unzip -q -n /content/val2017.zip -d /content/coco
COCO_CALIB = "/content/coco_calib"
os.makedirs(COCO_CALIB, exist_ok=True)
for f in sorted(glob.glob("/content/coco/val2017/*.jpg"))[:1024]:
    shutil.copy(f, COCO_CALIB)
print(len(os.listdir(COCO_CALIB)), "COCO calibration images")

In [ ]:
os.makedirs("/content/stock", exist_ok=True)
%cd /content/stock
!hailomz compile yolov8n --hw-arch hailo10h --calib-path /content/coco_calib 2>&1 | tee /content/stock/compile.log | tail -n 25
hefs = glob.glob("/content/stock/*.hef")
assert hefs, "no HEF: read /content/stock/compile.log"
shutil.copy(hefs[0], f"{OUT}/yolov8n_stock_dfc{DFC}.hef")
shutil.copy("/content/stock/compile.log", f"{OUT}/stock_compile.log")
print(sorted(os.listdir(OUT)))

**On the Pi (go/no-go 1, by Sat 7 Nov).** Copy both HEFs over, then:

```
hailortcli fw-control identify                  # Device Architecture HAILO10H, firmware 5.1.1
hailortcli parse-hef yolov8n_stock_dfc5.1.0.hef
hailortcli run yolov8n_stock_dfc5.1.0.hef
```

Then run `ayas.npu.HailoDetector(hef, n_classes=80)` on a 640 x 640 photo (`hailo/README.md`). If the reference HEF works and the self-compiled one does not, the compile chain is broken: fix it before compiling the custom model.

## 4. The Ayas model (6 steel classes: scratch, dent, pit, corrosion, crack, stain)

Inputs and checks: the ONNX must be 1 x 3 x 640 x 640 with a (1, 10, N) output (4 box rows + 6 classes, no NMS), and contain the six head convolutions where the Model Zoo yolov8n.yaml cuts the graph.

In [ ]:
import ast

CKPT = f"{WORK}/best.onnx"
CALIB_ZIP = f"{WORK}/calib_640.zip"
assert os.path.exists(CKPT), CKPT
assert os.path.exists(CALIB_ZIP), CALIB_ZIP
!rm -rf /content/calib_640 && unzip -q "{CALIB_ZIP}" -d /content
CALIB_DIR = "/content/calib_640"
n_calib = len(glob.glob(f"{CALIB_DIR}/*.png"))
print(n_calib, "calibration tiles")
assert n_calib >= 1024, "need at least 1024 640 px TRAIN tiles (training/calib_set.py)"

import onnx

m = onnx.load(CKPT)
dims = [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim]
assert dims == [1, 3, 640, 640], dims
out_dims = [d.dim_value for d in m.graph.output[0].type.tensor_type.shape.dim]
assert out_dims[:2] == [1, 4 + 6], out_dims
meta = {p.key: p.value for p in m.metadata_props}
CLASS_NAMES = list(ast.literal_eval(meta["names"]).values()) if "names" in meta else None
print("classes", CLASS_NAMES)
assert CLASS_NAMES in (None, ["scratch", "dent", "pit", "corrosion", "crack", "stain"]), CLASS_NAMES
END_NODES = [f"/model.22/cv{k}.{i}/cv{k}.{i}.2/Conv" for i in range(3) for k in (2, 3)]
names = {n.name for n in m.graph.node}
missing = [e for e in END_NODES if e not in names]
assert not missing, f"the Model Zoo yolov8n.yaml cuts the graph at these nodes: {missing}"
print("ONNX ok", dims, out_dims, "opset", [o.version for o in m.opset_import])

**Config copies.** `yolov8n.yaml` and its `.alls` are copied *next to the originals* inside `hailo_model_zoo/cfg/`, because the paths inside them (base yaml, NMS config json) are relative. The copied `.alls` gets `model_optimization_config(calibration, calibset_size=1024)`. If an NMS config is reported MISSING, the cell fetches it from the model's zip (`paths.url` in the yaml).

In [ ]:
import pathlib
import re
import zipfile

CFG = pathlib.Path("/content/hailo_model_zoo/hailo_model_zoo/cfg")
NEW = "ayas_yolov8n"
net_yaml = CFG / "networks" / "yolov8n.yaml"
text = net_yaml.read_text()
alls_name = re.search(r"alls_script:\s*(\S+)", text).group(1)
alls_files = sorted(CFG.glob(f"alls/**/{alls_name}"))
print("yaml:", net_yaml, "alls:", *alls_files, sep="\n")
assert alls_files, alls_name

NEW_YAML = net_yaml.with_name(f"{NEW}.yaml")
NEW_YAML.write_text(re.sub(r"(alls_script:\s*)\S+", rf"\g<1>{NEW}.alls", text))
NEW_ALLS = []
for a in alls_files:
    lines = [ln for ln in a.read_text().splitlines() if not ln.startswith("model_optimization_config(calibration")]
    lines.append("model_optimization_config(calibration, calibset_size=1024)")
    new = a.with_name(f"{NEW}.alls")
    new.write_text("\n".join(lines) + "\n")
    NEW_ALLS.append(new)
    nms = re.search(r'nms_postprocess\("([^"]+)"', new.read_text())
    if nms:
        target = (a.parent / nms.group(1)).resolve()
        print(a.parent.name, "NMS config:", target, "ok" if target.exists() else "MISSING")
        url = re.search(r"url:\s*(\S+)", text)
        if not target.exists() and url:
            urllib.request.urlretrieve(url.group(1), "/content/yolov8n_mz.zip")
            with zipfile.ZipFile("/content/yolov8n_mz.zip") as z:
                js = [n for n in z.namelist() if n.endswith(target.name)]
                assert js, f"{target.name} not in {url.group(1)}"
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(js[0]))
            print("fetched", target)
print(NEW_YAML.read_text())
print(NEW_ALLS[0].read_text())

**Compile.** First pass without `--performance` (it can add hours). `--classes 6` rewrites the NMS config for our classes. Expect tens of minutes on a T4.

In [ ]:
os.makedirs("/content/custom", exist_ok=True)
%cd /content/custom
!hailomz compile --ckpt "{CKPT}" --calib-path "{CALIB_DIR}" --yaml "{NEW_YAML}" --classes 6 --hw-arch hailo10h 2>&1 | tee /content/custom/compile.log | tail -n 40

**Quantisation check: SNR of every output at least 10 dB.** The DFC prints one `SNR` line per output layer at the end of optimisation. If there are none, run the layer noise analysis below. If an output is under 10 dB, add `quantization_param(<that output layer>, precision_mode=a16_w16)` to the copied `.alls` and compile again (`a16_w8` is reported to crash on hailo10h).

In [ ]:
log = open("/content/custom/compile.log").read()
snr = [(m.group(1), float(m.group(2))) for m in re.finditer(r"(\S+)\s+SNR:?\s*(-?[\d.]+)\s*dB", log)]
print(*snr, sep="\n")
if not snr:
    print("No SNR lines in the log: run the noise analysis cell below before trusting this HEF.")
low = [s for s in snr if s[1] < 10.0]
assert not low, f"outputs below 10 dB: {low}"

In [ ]:
# Optional layer noise analysis. Check the exact flags first with: !hailo analyze-noise --help
# (not verified for DFC 5.1.0 here). It needs the optimised (quantised) HAR and calibration data.
hars = sorted(glob.glob("/content/custom/*.har"))
print(hars)
# !hailo analyze-noise {hars[0]} --data-path {CALIB_DIR} --batch-size 8 --data-count 64

## 5. Save to Drive

In [ ]:
import datetime
import json
import platform

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dst = f"{OUT}/ayas_{stamp}"
os.makedirs(dst, exist_ok=True)
hefs = glob.glob("/content/custom/*.hef")
assert hefs, "no HEF: read /content/custom/compile.log"
for f in hefs + glob.glob("/content/custom/*.har") + ["/content/custom/compile.log", str(NEW_YAML)]:
    shutil.copy(f, dst)
for a in NEW_ALLS:
    shutil.copy(a, f"{dst}/{a.parent.name}_{a.name}")
mz = subprocess.run(["git", "-C", "/content/hailo_model_zoo", "rev-parse", "HEAD"], capture_output=True, text=True)
info = {"dfc": DFC, "model_zoo": "v5.1.0", "model_zoo_commit": mz.stdout.strip(), "python": platform.python_version(),
        "gpu": gpu.stdout.strip(), "classes": CLASS_NAMES or ["scratch", "dent", "pit", "corrosion", "crack", "stain"], "n_classes": 6, "calib_tiles": n_calib, "snr_db": snr, "ckpt": CKPT,
        "hw_arch": "hailo10h", "performance_flag": False}
with open(f"{dst}/compile_info.json", "w") as f:
    json.dump(info, f, indent=2)
for h in hefs:      # ayas.npu.HailoDetector reads the class names from <hef stem>.json next to the HEF
    with open(f"{dst}/{os.path.splitext(os.path.basename(h))[0]}.json", "w") as f:
        json.dump({"classes": info["classes"], "imgsz": 640, "source": CKPT}, f, indent=2)
print(dst, sorted(os.listdir(dst)))

## 6. On the Pi (go/no-go 2, Fri 13 Nov)

1. `hailortcli parse-hef <hef>` and `hailortcli run <hef>` must pass.
2. Run the same test grids through `ayas.npu.HailoDetector` and `OnnxDetector` (`training/eval_plates.py --model <hef|onnx>`); the HEF's mAP on TRAIN-val must be within 5 points of ONNX FP32.
3. If not: ship YOLO on the CPU (ONNX Runtime or NCNN), stop Hailo work, and say so in the report. See `hailo/README.md`.